# test_nodes

用于直接验证 nodes 的真实链路。

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'src' / 'task_router_graph').exists()
)
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from task_router_graph.graph import TaskRouterGraph
from task_router_graph.nodes import route_node, execute_node, update_node
from task_router_graph.schema import Environment

print('模块加载完成')
print('已检测 API_KEY_Qwen：', bool(os.getenv('API_KEY_Qwen')))


In [ ]:
graph = TaskRouterGraph(config_path='configs/graph.yaml')
env = Environment()
run_root = (graph.root / graph.config['paths']['run_root']).resolve()

task, trace = route_node(
    llm=graph._llm,
    controller_system=graph._controller_system,
    controller_skills_index=graph._controller_skills_index,
    environment=env,
    user_input='请帮我做一次 anthropic_ver_1 的功能测试',
    workspace_root=graph.root,
    run_root=run_root,
    max_steps=graph._max_controller_steps,
)

print('路由完成')
print('任务类型：', task.type)
print('动作条数：', len(trace))
print('最后动作：', trace[-1].action_kind)


In [ ]:
task2, reply = execute_node(
    llm=graph._llm,
    normal_system=graph._normal_system,
    normal_skills_index=graph._normal_skills_index,
    environment=env,
    task=task,
)

env = update_node(env, '请帮我做一次 anthropic_ver_1 的功能测试', trace, task2, reply)

print('执行与回写完成')
print('任务状态：', task2.status)
print('任务结果：', task2.result)
print('回复内容：', reply)
print('环境轮次：', len(env.rounds))
